# Speech Emotion Recognition — Step-by-Step Walkthrough

This notebook walks through the full project end to end, with explanations
at each step. Use this for your presentation/demo, or to understand the
pipeline before reading the standalone scripts in `src/`.

**Pipeline:** Raw audio → Feature extraction (MFCC + Chroma + Mel + ZCR + RMS)
→ Data augmentation → CNN + BiLSTM + Attention model → Evaluation → Live prediction.

Make sure you've downloaded RAVDESS into `../data/raw/` before running this
(see the main README.md for the download link).

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt

%matplotlib inline

## 1. Load and visualize one sample clip

Let's look at what raw speech audio actually looks like, and how MFCCs
summarize it.

In [ ]:
from feature_extraction import load_audio, extract_features_sequence, SAMPLE_RATE

# Point this at any .wav file from your downloaded dataset
sample_path = "../data/raw/Actor_01/03-01-03-01-01-01-01.wav"  # 03 = happy

y, sr = load_audio(sample_path)

fig, axes = plt.subplots(2, 1, figsize=(10, 6))
librosa.display.waveshow(y, sr=sr, ax=axes[0])
axes[0].set_title("Raw waveform")

mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
img = librosa.display.specshow(mfcc, x_axis="time", ax=axes[1])
axes[1].set_title("MFCC (what the model actually 'sees')")
fig.colorbar(img, ax=axes[1])
plt.tight_layout()
plt.show()

print("Raw waveform length:", y.shape, " -> MFCC shape:", mfcc.shape)
print("MFCC compresses ~66,000 raw samples into a much smaller, structured representation.")

## 2. See what augmentation does to a clip

This is one of the "uniqueness" additions — augmentation creates realistic
variations so the model doesn't just memorize individual speakers.

In [ ]:
from augmentation import augment_sample
import IPython.display as ipd

variants = augment_sample(y, sr)

fig, axes = plt.subplots(len(variants), 1, figsize=(10, 2*len(variants)), sharex=True)
for ax, (name, wav) in zip(axes, variants.items()):
    librosa.display.waveshow(wav, sr=sr, ax=ax)
    ax.set_ylabel(name, rotation=0, labelpad=40, fontsize=9)
plt.tight_layout()
plt.show()

# Listen to the original vs. an augmented version
print("Original:")
ipd.display(ipd.Audio(variants["original"], rate=sr))
print("With pitch shifted up:")
ipd.display(ipd.Audio(variants["pitch_up"], rate=sr))

## 3. Build the full dataset (features + augmentation for every clip)

This step can take a while for the full RAVDESS dataset (~1,440 clips x 6
variants each = ~8,600 samples). It only needs to be run once — results are
cached to `data/processed/`.

In [ ]:
from dataset import build_dataset

RAW_DIR = "../data/raw"
OUT_DIR = "../data/processed"

X, y_labels = build_dataset(RAW_DIR, OUT_DIR, use_augmentation=True)
print("Final dataset shape:", X.shape, "labels:", set(y_labels))

## 4. Look at the model architecture

CNN layers -> Bidirectional LSTM -> Attention -> Dense classifier.
See the docstring in `src/model.py` for the reasoning behind each part.

In [ ]:
from model import build_model

model = build_model(input_shape=X.shape[1:], num_classes=len(set(y_labels)))
model.summary()

## 5. Train the model

For the full training loop with callbacks, checkpointing, and evaluation
plots, run the standalone script instead of duplicating it here:

```bash
cd ../src
python train.py
```

This keeps the notebook fast to re-run while still letting you inspect
every stage above interactively.

## 6. Predict on a new clip

Once `train.py` has produced `models/ser_model.keras`, you can predict on
any new audio file: 

In [ ]:
from predict import predict_emotion

label, probs = predict_emotion(sample_path)
print("Predicted emotion:", label)
for emo, p in sorted(probs.items(), key=lambda x: -x[1]):
    print(f"  {emo:12s}: {p*100:.2f}%")

plt.figure(figsize=(6,3))
plt.bar(probs.keys(), probs.values(), color="#6C63FF")
plt.xticks(rotation=45)
plt.title(f"Predicted: {label}")
plt.tight_layout()
plt.show()

## 7. Next steps for the live demo

Run the interactive Streamlit app to upload/record clips and see live
predictions with a confidence chart:

```bash
cd ..
streamlit run app/streamlit_app.py
```

This is the best part to show during your internship presentation.